In [1]:
using Revise

In [2]:
using GeneRegulatorySystems
using GeneRegulatorySystems.Models
using GeneRegulatorySystems.Models.NetworkRepresentation
using GeneRegulatorySystems.Models.V1
using GeneRegulatorySystems.Scheduling
import JSON
using GarishPrint

macro pp(x) :(GarishPrint.pprint($(esc(x)))) end;

Load schedule

In [3]:
path_diff = "examples/specification/differentiation.schedule.json"
path_v1 = "examples/toy/ACDC.schedule.json"
path_kronecker = "examples/benchmark/kronecker-small.schedule.json"
path_rand_diff = "examples/specification/random-differentiation.schedule.json"

schedule! = Models.load(path_v1, seed="seed123");

In [ ]:
f! = Scheduling.reify(schedule!, "-2-1+-1+")

In [ ]:
rs = f!.f!.model.model.definition

In [ ]:
using Catalyst
using GraphMakie
using NetworkLayout
using CairoMakie

Catalyst.plot_network(rs)

Reify the schedule. Need to do a dryrun to get all the primitives

In [ ]:
# this only extracts the first network that it finds in the schedule, rather than all of them
function extract_network(schedule::Scheduling.Schedule)
    network = nothing
    
    function dryrun_collector(primitive!, x, Δt; path, _...)
        if !(isfinite(Δt) && Δt > 0.0)
            return
        end
        if network === nothing
            network = Models.NetworkRepresentation.entity(primitive!)
        end
    end
    
    schedule(Models.FlatState(); dryrun=dryrun_collector)
    return network
end

network = extract_network(schedule!)

In [ ]:
schedule_files = [
    "examples/specification/differentiation.schedule.json",
    "examples/benchmark/differentiated-small.schedule.json",
    "examples/benchmark/differentiated-medium.schedule.json",
    "examples/benchmark/kronecker-tiny.schedule.json",
    "examples/benchmark/kronecker-small.schedule.json",
    "examples/benchmark/kronecker-medium.schedule.json",
    "examples/benchmark/kronecker-big.schedule.json",
    #"examples/benchmark/kronecker-huge.schedule.json",
]

results = Dict()
timings = Dict()

for path in schedule_files
    try
        @info "Processing: $path"
        elapsed = @elapsed begin
            sched = Models.load(path, seed="seed123")
            net = extract_network(sched)
        end
        
        if net !== nothing
            flat_nodes, flat_links = NetworkRepresentation.flatten(net)
            num_nodes = length(flat_nodes)
            num_links = length(flat_links)
            @info "  Success: $(num_nodes) nodes, $(num_links) links ($(round(elapsed, digits=3))s)"
            results[path] = "Success (nodes=$num_nodes, links=$num_links)"
            timings[path] = (time=elapsed, nodes=num_nodes)
        else
            @warn "  Failed: network is nothing"
            results[path] = "Failed"
            timings[path] = (time=elapsed, nodes=0)
        end
    catch e
        @error "  Error: $e"
        results[path] = "Error: $(e)"
        timings[path] = (time=NaN, nodes=0)
    end
end

In [ ]:
using CairoMakie

nodes_list = [timings[path].nodes for path in schedule_files if !isnan(timings[path].time)]
times_list = [timings[path].time for path in schedule_files if !isnan(timings[path].time)]
labels_list = [splitdir(path)[2] for path in schedule_files if !isnan(timings[path].time)]

fig = Figure(size=(1000, 600))
ax = Axis(fig[1, 1], xlabel="Number of Nodes", ylabel="Time (s)", 
          title="Processing Time vs Network Size")

scatter!(ax, nodes_list, times_list, markersize=10, color=:blue, alpha=0.6)

for (x, y, label) in zip(nodes_list, times_list, labels_list)
    text!(ax, x, y + 0.01, text=label, fontsize=9)
end

fig

In [ ]:
using GeneRegulatorySystems.Models: Wrapped, Instant, Label, Descriptions

_label(wrapped::Wrapped) = _label(Models.describe(wrapped.definition))

_label(label::Label) = label.label

_label(model::Instant) = repr(model)

function _label(desc::Descriptions)
    i = findfirst(d -> d isa Label, desc.descriptions)
    if i !== nothing
        _label(desc.descriptions[i])
    else
        ""
    end
end

function get_segments(schedule::Scheduling.Schedule)
    segments = []
    function dryrun_collector(primitive!, x, Δt; path, _...)
        label = _label(primitive!.f!.model)
        maybe_source = primitive!.path == path ? "" : " ($(primitive!.path))"
        push!(segments, (path = path, from = x.t, to = x.t + Δt, label = label, source=maybe_source))
    end
    
    schedule(Models.FlatState(); dryrun=dryrun_collector)
    return segments
end

segments = get_segments(schedule!)

In [ ]:
NetworkRepresentation.entity(Scheduling.reify(schedule!, ".do"))

# Exploring schedule editing via reify + representation

In [10]:
# Start with a simple V1 schedule
schedule_v1 = Models.load("examples/toy/repressilator.schedule.json", seed="seed123")

# What does the top-level schedule look like?
println("Type: ", typeof(schedule_v1))
println("Path: '$(schedule_v1.path)'")
println("Branch: $(schedule_v1.branch)")
println("Spec type: ", typeof(schedule_v1.specification))
println("Bindings keys: ", keys(schedule_v1.bindings))

Type: Schedule{GeneRegulatorySystems.Specifications.Scope}
Path: ''
Branch: false
Spec type: GeneRegulatorySystems.Specifications.Scope
Bindings keys: [:into, :channel, :defaults, :seed]


In [3]:
# Collect all primitives via dryrun to see their paths
function collect_primitives(schedule::Scheduling.Schedule)
    primitives = []
    function collector(primitive!, x, dt; path, _...)
        push!(primitives, (
            path = path,
            primitive_path = primitive!.path,
            model_type = typeof(primitive!.f!),
            dt = dt,
        ))
    end
    schedule(Models.FlatState(); dryrun = collector)
    return primitives
end

# for p in collect_primitives(schedule_v1)
#     println("path=$(p.path)  primitive_path=$(p.primitive_path)  type=$(p.model_type)  dt=$(p.dt)")
# end

collect_primitives (generic function with 1 method)

In [4]:
# Now reify to the V1 model using primitive_path from the dryrun
# For repressilator, the model is bound to "do" in a Scope, so the path is ".do"
wrapped = Scheduling.reify(schedule_v1, "+.do")
println("Type: ", typeof(wrapped))
println("Primitive path: ", wrapped.definition)

# Get the V1.Definition
definition = wrapped.model.definition
println("\nV1 Definition type: ", typeof(definition))
println("Number of genes: ", length(definition.genes))
for (i, g) in enumerate(definition.genes)
    println("  Gene $i: name=$(g.name), activation=$(g.activation.slots), repression=$(g.repression.slots)")
end

UndefVarError: UndefVarError: `schedule_v1` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [5]:
# Round-trip: representation() on the Definition gives us back the spec dict
using GeneRegulatorySystems.Specifications: representation

spec_dict = representation(definition)
@pp spec_dict

# Compare with the original JSON
println("\n--- Original JSON model block ---")
original = JSON.parsefile("examples/toy/repressilator.schedule.json", dicttype=Dict{Symbol, Any})
@pp original[:step][:do]

UndefVarError: UndefVarError: `definition` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
Hint: a global variable of this name also exists in CodeTracking.

In [14]:
# Now let's MODIFY the definition: add a 4th gene with default rates and an activation edge from gene 1
defaults = Models.load_defaults()
default_base_rates = Specifications.cast(V1.EukaryoteBaseRates, defaults[:gene][:base_rates])

new_gene = V1.Gene(
    name = Symbol("4"),
    base_rates = default_base_rates,
    activation = V1.Activation(slots = [V1.HillRegulator(from = Symbol("1"), at = 5.0)]),
)

# Build modified definition
modified_definition = V1.Definition(
    genes = vcat(definition.genes, [new_gene]),
    polymerases = definition.polymerases,
    ribosomes = definition.ribosomes,
    proteasomes = definition.proteasomes,
    reactions = definition.reactions,
)

println("Modified definition has $(length(modified_definition.genes)) genes")
@pp representation(modified_definition)

Modified definition has 4 genes
Dict{Symbol, Any} with 1 entry:
  Symbol("{regulation/v1}") => Dict{Symbol, Any}(:genes => Dict{Symbol, Any}[Dict(:activation => Dict{Symbol, Any}[Dict(:from => "1", :at => 2.0)], :repression => Dict{Symbol, Any}[Dict(:from => "3", :at => 4.0)], :base_rates => Dict{Symbol, Any}(:activation => 2.5, :processing => 0.02, :transcription => 0.001, :protein_decay => 3.0e-10, :trigger => 6.6e-7, :mrna_decay => 0.001, :deactivation => 10.0, :translation => 2.5e-9, :abortion => 0.01, :premrna_decay => 0.001), :name => "1"), Dict(:activation => Dict{Symbol, Any}[Dict(:from => "2", :at => 2.0)], :repression => Dict{Symbol, Any}[Dict(:from => "1", :at => 4.0)], :base_rates => Dict{Symbol, Any}(:activation => 2.5, :processing => 0.02, :transcription => 0.001, :protein_decay => 3.0e-10, :trigger => 6.6e-7, :mrna_decay => 0.001, :deactivation => 10.0, :translation => 2.5e-9, :abortion => 0.01, :premrna_decay => 0.001), :name => "2"), Dict(:activation => Dict{Symbol, An

In [15]:
# KEY QUESTION: can we splice this back into the spec?
# The primitive_path is ".do" -- meaning the model lives at spec["do"] in the Scope's definitions
# Let's see if we can navigate the raw spec dict using the path tokens

# Load raw spec dict
raw_spec = JSON.parsefile("examples/toy/repressilator.schedule.json", dicttype=Dict{Symbol, Any})
println("Top-level keys: ", keys(raw_spec))
println("raw_spec[:step][:do] keys: ", keys(raw_spec[:step][:do]))

# The representation() output for V1.Definition wraps in {regulation/v1} key
modified_repr = representation(modified_definition)
println("\nModified repr keys: ", keys(modified_repr))

# So we'd replace raw_spec[:step][:do] with modified_repr
raw_spec[:step][:do] = modified_repr
println("\n--- Modified full spec ---")
println(JSON.json(raw_spec, 2))

Top-level keys: [:step, :label]
raw_spec[:step][:do] keys: [Symbol("{regulation/v1}")]

Modified repr keys: [Symbol("{regulation/v1}")]

--- Modified full spec ---
{
  "step": {
    "do": {
      "{regulation/v1}": {
        "genes": [
          {
            "activation": [
              {
                "from": "1",
                "at": 2.0
              }
            ],
            "repression": [
              {
                "from": "3",
                "at": 4.0
              }
            ],
            "base_rates": {
              "activation": 2.5,
              "processing": 0.02,
              "transcription": 0.001,
              "protein_decay": 3.0e-10,
              "trigger": 6.6e-7,
              "mrna_decay": 0.001,
              "deactivation": 10.0,
              "translation": 2.5e-9,
              "abortion": 0.01,
              "premrna_decay": 0.001
            },
            "name": "1"
          },
          {
            "activation": [
              {
 

In [16]:
# Verify the modified spec actually loads and works
modified_json = JSON.json(raw_spec)
modified_schedule = Models.parse(modified_json, seed="seed123")

# Reify again to check our new gene is there
wrapped2 = Scheduling.reify(modified_schedule, "+.do")
def2 = wrapped2.model.definition
println("Genes in modified schedule: $(length(def2.genes))")
for (i, g) in enumerate(def2.genes)
    println("  Gene $i: name=$(g.name), act=$(g.activation.slots), rep=$(g.repression.slots)")
end

Genes in modified schedule: 4
  Gene 1: name=1, act=GeneRegulatorySystems.Models.V1.HillRegulator[GeneRegulatorySystems.Models.V1.HillRegulator(Symbol("1"), 2.0, -1.0)], rep=GeneRegulatorySystems.Models.V1.HillRegulator[GeneRegulatorySystems.Models.V1.HillRegulator(Symbol("3"), 4.0, -1.0)]
  Gene 2: name=2, act=GeneRegulatorySystems.Models.V1.HillRegulator[GeneRegulatorySystems.Models.V1.HillRegulator(Symbol("2"), 2.0, -1.0)], rep=GeneRegulatorySystems.Models.V1.HillRegulator[GeneRegulatorySystems.Models.V1.HillRegulator(Symbol("1"), 4.0, -1.0)]
  Gene 3: name=3, act=GeneRegulatorySystems.Models.V1.HillRegulator[GeneRegulatorySystems.Models.V1.HillRegulator(Symbol("3"), 2.0, -1.0)], rep=GeneRegulatorySystems.Models.V1.HillRegulator[GeneRegulatorySystems.Models.V1.HillRegulator(Symbol("2"), 4.0, -1.0)]
  Gene 4: name=4, act=GeneRegulatorySystems.Models.V1.HillRegulator[GeneRegulatorySystems.Models.V1.HillRegulator(Symbol("1"), 5.0, -1.0)], rep=GeneRegulatorySystems.Models.V1.HillRegulat

In [17]:
# BUT: this approach requires knowing the spec dict structure (raw_spec[:step][:do]).
# The path ".do" means: descend into Scope definitions, key "do".
# Can reify help us navigate the raw dict too?
# 
# Actually, the issue is: representation() loses the "$" template references.
# The original spec uses {"$": ["defaults", "gene"]} but representation() 
# expands those into the full base_rates dict.
# 
# Let's verify this is a problem:
println("Original gene 1 spec (with \$ ref):")
@pp raw_spec[:step][:do][Symbol("{regulation/v1}")][:genes][1]

println("\nAfter representation (expanded, no \$ refs):")
@pp representation(modified_definition)[Symbol("{regulation/v1}")][:genes][1]

Original gene 1 spec (with $ ref):
Dict{Symbol, Any} with 4 entries:
  :activation => Dict{Symbol, Any}[Dict(:from => "1", :at => 2.0)]
  :repression => Dict{Symbol, Any}[Dict(:from => "3", :at => 4.0)]
  :base_rates => Dict{Symbol, Any}(:activation => 2.5, :processing => 0.02, :transcription => 0.001, :protein_decay => 3.0e-10, :trigger => 6.6e-7, :mrna_decay => 0.001, :deactivation => 10.0, :translation => 2.5e-9, :abortion => 0.01, :premrna_decay => 0.001)
  :name => "1"
After representation (expanded, no $ refs):
Dict{Symbol, Any} with 4 entries:
  :activation => Dict{Symbol, Any}[Dict(:from => "1", :at => 2.0)]
  :repression => Dict{Symbol, Any}[Dict(:from => "3", :at => 4.0)]
  :base_rates => Dict{Symbol, Any}(:activation => 2.5, :processing => 0.02, :transcription => 0.001, :protein_decay => 3.0e-10, :trigger => 6.6e-7, :mrna_decay => 0.001, :deactivation => 10.0, :translation => 2.5e-9, :abortion => 0.01, :premrna_decay => 0.001)
  :name => "1"

In [18]:
# Alternative approach: instead of replacing the whole model block,
# just modify the genes array IN the raw spec dict.
# We know model_path tells us WHERE the {regulation/v1} block is.
# We can navigate there, then surgically edit just the genes array.

# Let's try: navigate raw spec by model_path to find the genes array
raw_spec2 = JSON.parsefile("examples/toy/repressilator.schedule.json", dicttype=Dict{Symbol, Any})

# The model is at raw_spec[:step][:do][Symbol("{regulation/v1}")]
v1_block = raw_spec2[:step][:do][Symbol("{regulation/v1}")]
genes_array = v1_block[:genes]

# Add a new gene -- just append the dict form
new_gene_dict = Dict{Symbol, Any}(
    Symbol("\$") => ["defaults", "gene"],  # use the $ reference for base_rates!
    :activation => [Dict{Symbol, Any}(:from => 1, :at => 5.0)],
)
push!(genes_array, new_gene_dict)

# This preserves all existing $ references in other genes!
println("Modified genes array:")
for (i, g) in enumerate(genes_array)
    @pp g
    println()
end

# Verify it loads
modified_json2 = JSON.json(raw_spec2)
sched2 = Models.parse(modified_json2, seed="seed123")
wrapped3 = Scheduling.reify(sched2, "+.do")
println("Genes: $(length(wrapped3.model.definition.genes))")
for g in wrapped3.model.definition.genes
    println("  $(g.name): act=$(g.activation.slots) rep=$(g.repression.slots)")
end

Modified genes array:
Dict{Symbol, Any} with 3 entries:
  :activation => Any[Dict{Symbol, Any}(:from => 1, :at => 2.0)]
  :repression => Any[Dict{Symbol, Any}(:from => 3, :at => 4.0)]
  :$ => Any["defaults", "gene"]
Dict{Symbol, Any} with 3 entries:
  :activation => Any[Dict{Symbol, Any}(:from => 2, :at => 2.0)]
  :repression => Any[Dict{Symbol, Any}(:from => 1, :at => 4.0)]
  :$ => Any["defaults", "gene"]
Dict{Symbol, Any} with 3 entries:
  :activation => Any[Dict{Symbol, Any}(:from => 3, :at => 2.0)]
  :repression => Any[Dict{Symbol, Any}(:from => 2, :at => 4.0)]
  :$ => Any["defaults", "gene"]
Dict{Symbol, Any} with 2 entries:
  :activation => Dict{Symbol, Any}[Dict(:from => 1, :at => 5.0)]
  :$ => ["defaults", "gene"]
Genes: 4
  1: act=GeneRegulatorySystems.Models.V1.HillRegulator[GeneRegulatorySystems.Models.V1.HillRegulator(Symbol("1"), 2.0, -1.0)] rep=GeneRegulatorySystems.Models.V1.HillRegulator[GeneRegulatorySystems.Models.V1.HillRegulator(Symbol("3"), 4.0, -1.0)]
  2: act=Gen

In [ ]:
# Now the big question: how to navigate the raw spec dict by model_path?
# Let's understand what model_path values look like for different schedules

for path in ["examples/toy/repressilator.schedule.json",
             "examples/specification/simple.schedule.json",
             "examples/specification/differentiation.schedule.json",
             "examples/toy/ACDC.schedule.json"]
    println("\n=== $path ===")
    sched = Models.load(path, seed="seed123")
    for p in collect_primitives(sched)
        isfinite(p.dt) && p.dt > 0.0 || continue
        println("  execution_path=$(p.path)  model_path=$(p.primitive_path)")
    end
end

In [5]:
# Gene name extraction without full network -- using multiple dispatch
# The pattern from network_representation.jl: dispatch on definition type


gene_names(definition::V1.Definition, ::Models.Wrapped) = [g.name for g in definition.genes]
gene_names(wrapped::Models.Wrapped) = gene_names(wrapped.definition, wrapped)
# Fallback: descend into wrapped model
gene_names(definition, wrapped::Models.Wrapped) = gene_names(wrapped.model)
gene_names(primitive::Scheduling.Primitive) = gene_names(primitive.f!)

# Test it
wrapped = Scheduling.reify(schedule_v1, "+.do")
gene_names(wrapped)

UndefVarError: UndefVarError: `schedule_v1` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

# Surgical spec editing via path-based navigation

Navigate a raw spec dict using the same path tokens as `Scheduling.reify`,
then surgically edit the model block in-place (preserving `$` references).

In [4]:
# Explore using reify directly instead of reimplementing navigation on raw dicts.
# reify already handles all path logic — what can we get from the reified result?

for path in ["examples/toy/repressilator.schedule.json",
             "examples/toy/ACDC.schedule.json",
             "examples/toy/cascade.schedule.json",
             "examples/toy/repressilator-entrained.schedule.json"]
    println("\n=== $(basename(path)) ===")
    sched = Models.load(path, seed="seed123")
    mp = first(collect_model_paths(sched))
    println("  model_path: $mp")

    # reify to the primitive
    result = Scheduling.reify(sched, mp)
    println("  reify result type: $(typeof(result))")
    println("  fieldnames: $(fieldnames(typeof(result)))")

    if result isa Scheduling.Primitive
        println("  primitive.path: $(result.path)")
        println("  primitive.bindings keys: $(keys(result.bindings))")
        println("  primitive.f! type: $(typeof(result.f!))")
        # What about the model inside?
        f = Models.unwrap(result.f!)
        println("  unwrapped model type: $(typeof(f))")
        if hasproperty(f, :definition)
            println("  definition type: $(typeof(f.definition))")
        end
    end

    # What if we reify one step LESS — stop at the Schedule that holds the model?
    # For "+.do", try just "+"
    parts = split(mp, ".")
    if length(parts) > 1
        prefix = parts[1]  # e.g. "+"
        partial = Scheduling.reify(sched, prefix)
        println("\n  partial reify('$prefix') type: $(typeof(partial))")
        println("  fieldnames: $(fieldnames(typeof(partial)))")
        if partial isa Scheduling.Schedule
            println("  specification type: $(typeof(partial.specification))")
            println("  bindings keys: $(keys(partial.bindings))")
        end
    end

    # What about reifying to just before the model — the bindings dict should
    # contain the raw spec dict we need
    # For repressilator path "+.do", reify(sched, "+") gives us a Schedule
    # whose bindings should contain :do => the raw model dict
    println()
end


=== repressilator.schedule.json ===


UndefVarError: UndefVarError: `collect_model_paths` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [33]:
# The partial reify gives us a Schedule{Scope} whose .specification holds
# the parsed Scope. Do the Scope definitions / Template values reference
# the original raw dict objects from JSON parsing?

sched = Models.load("examples/toy/repressilator.schedule.json", seed="seed123")
scope_sched = Scheduling.reify(sched, "+")

scope = scope_sched.specification
println("Scope definitions keys: ", keys(scope.definitions))
println()

# The :do definition should be a Template whose .value is the raw {regulation/v1} dict
do_template = scope.definitions[:do]
println("do_template type: ", typeof(do_template))
println("do_template fields: ", fieldnames(typeof(do_template)))
println("do_template.value type: ", typeof(do_template.value))
println("do_template.value: ")
@pp do_template.value

# Is this the same object as what's in the raw JSON?
raw = JSON.parsefile("examples/toy/repressilator.schedule.json", dicttype=Dict{Symbol, Any})
println("\nraw[:step][:do]:")
@pp raw[:step][:do]

# They won't be === (different parse), but do they have the same structure?
# More importantly: can we MUTATE do_template.value to edit the schedule?
println("\nCan we find genes via template.value?")
v1_key = Symbol("{regulation/v1}")
if haskey(do_template.value, v1_key)
    genes = do_template.value[v1_key][:genes]
    println("Found $(length(genes)) genes in template.value")
    for (i, g) in enumerate(genes)
        println("  Gene $i keys: ", keys(g))
    end
end

Scope definitions keys: [:do]

do_template type: GeneRegulatorySystems.Specifications.Template
do_template fields: (:value, :constructor, :free)
do_template.value type: Dict{Symbol, Any}
do_template.value: 
Dict{Symbol, Any} with 1 entry:
  :genes => Any[Dict{Symbol, Any}(:activation => Any[Dict{Symbol, Any}(:from => 1, :at => 2.0)], :repression => Any[Dict{Symbol, Any}(:from => 3, :at => 4.0)], :$ => Any["defaults", "gene"]), Dict{Symbol, Any}(:activation => Any[Dict{Symbol, Any}(:from => 2, :at => 2.0)], :repression => Any[Dict{Symbol, Any}(:from => 1, :at => 4.0)], :$ => Any["defaults", "gene"]), Dict{Symbol, Any}(:activation => Any[Dict{Symbol, Any}(:from => 3, :at => 2.0)], :repression => Any[Dict{Symbol, Any}(:from => 2, :at => 4.0)], :$ => Any["defaults", "gene"])]
raw[:step][:do]:
Dict{Symbol, Any} with 1 entry:
  Symbol("{regulation/v1}") => Dict{Symbol, Any}(:genes => Any[Dict{Symbol, Any}(:activation => Any[Dict{Symbol, Any}(:from => 1, :at => 2.0)], :repression => Any[Dict{

In [8]:
using GeneRegulatorySystems.Specifications: pluck

# Reserved keys per spec type (mirroring maybe_scope / maybe_each in specifications.jl)
const SCOPE_RESERVED = Set([:step, :branch])
const EACH_RESERVED = Set([:each, :as, :step, :branch])

"""
    scope_definitions(spec) -> Dict{Symbol, Any}

Extract definitions from a raw Scope dict — everything except the reserved keys.
Mirrors `maybe_scope` / `maybe_each` in specifications.jl.
"""
function scope_definitions(spec::AbstractDict{Symbol})
    reserved = haskey(spec, :each) ? EACH_RESERVED : SCOPE_RESERVED
    Dict{Symbol, Any}(k => v for (k, v) in spec if k ∉ reserved)
end

"""
    navigate_spec(spec, path::AbstractString; bindings=Dict{Symbol,Any}())
    -> (target, bindings)

Navigate a raw spec dict following a model_path string, accumulating scope
bindings along the way (mirroring `evaluate_bindings` in scheduling.jl).

Returns both the target dict AND the accumulated bindings, so that `\$`
references in the target can be resolved via `pluck(bindings, ref)`.

Path token semantics (mirroring `Scheduling.reify`):
- `+` or `/` on a Scope/Each -> merge definitions into bindings, descend into step
- `.key` -> look up in current bindings (definition or outer binding)
- `-N` or `N` on a List (Vector) -> index into items
- `-N` or `N` on an Each -> descend into step (skip iteration index)

In raw JSON a Scope wrapping an Each is a single dict with both scope keys and
`:each`. `+`/`/` handles only the Scope level (merge defs, stay on same dict);
the digit token then handles the Each level (descend into `:step`).
"""
function navigate_spec(spec, path::AbstractString; bindings=Dict{Symbol,Any}())
    isempty(path) && return (spec, bindings)

    if spec isa AbstractVector
        m = match(r"^-?(\d+)(.*)", path)
        m !== nothing || error("Expected integer index for Vector, got: '$path'")
        idx = parse(Int, m[1])
        return navigate_spec(spec[idx], String(m[2]); bindings)
    end

    token = path[1]

    if token == '+' || token == '/'
        tail = path[2:end]
        merged = merge(bindings, scope_definitions(spec))
        if haskey(spec, :each)
            # Scope+Each combined in raw JSON — handle only the Scope level,
            # stay on same dict so the next digit token handles the Each.
            return navigate_spec(spec, tail; bindings=merged)
        else
            # Plain Scope — descend into :step
            return navigate_spec(spec[:step], tail; bindings=merged)
        end
    elseif token == '.'
        m = match(r"^\.([^\.\+\/]+)(.*)", path)
        m !== nothing || error("Invalid binding key after '.': '$path'")
        key = Symbol(m[1])
        value = get(bindings, key) do
            spec[key]
        end
        return navigate_spec(value, String(m[2]); bindings)
    elseif token == '-' || isdigit(token)
        m = match(r"^-?(\d+)(.*)", path)
        m !== nothing || error("Expected integer at: '$path'")
        haskey(spec, :each) || error("Integer token but no :each key. Keys: $(keys(spec))")
        merged = merge(bindings, scope_definitions(spec))
        return navigate_spec(spec[:step], String(m[2]); bindings=merged)
    else
        error("Unknown path token at: '$path'")
    end
end

"""
    resolve_dollar(v1_block, bindings) -> Dict with :genes

Given a V1 model block (the value under the `{regulation/v1}` key),
resolve `\$` references using `pluck` — exactly as `substituted()` does
in specifications.jl.

Returns the raw dict that actually contains the `:genes` array, which may
be the prototype (shared base model) when using `\$` inheritance.
"""
function resolve_dollar(block::AbstractDict{Symbol}, bindings::AbstractDict{Symbol})
    if haskey(block, Symbol("\$"))
        prototype = pluck(bindings, block[Symbol("\$")])
        if prototype isa AbstractDict{Symbol}
            return resolve_dollar(prototype, bindings)
        end
        return prototype
    end
    return block
end

resolve_dollar

In [35]:
# Test navigate_spec on multiple schedule files
test_schedules = [
    "examples/toy/repressilator.schedule.json",
    "examples/specification/simple.schedule.json",
    "examples/toy/ACDC.schedule.json",
    "examples/toy/ACDC-simple.schedule.json",
    "examples/specification/differentiation.schedule.json",
    "examples/toy/switch.schedule.json",
    "examples/toy/cascade.schedule.json",
    "examples/toy/coupled-repressilator.schedule.json",
]

for path in test_schedules
    println("\n=== $(basename(path)) ===")

    # Collect unique model_paths from dryrun
    sched = Models.load(path, seed="seed123")
    model_paths = Set{String}()
    for p in collect_primitives(sched)
        isfinite(p.dt) && p.dt > 0.0 || continue
        push!(model_paths, p.primitive_path)
    end

    # Parse raw JSON
    raw = JSON.parsefile(path, dicttype=Dict{Symbol, Any})

    for mp in sort(collect(model_paths))
        # Navigate raw dict (now returns tuple)
        target, bindings = navigate_spec(raw, mp)

        # Check for tagged template key {model_type}
        tagged = filter(k -> occursin(r"\{.*\}", String(k)), collect(keys(target)))

        # Cross-check: reify gives a model with the same definition type
        wrapped = Scheduling.reify(sched, mp)
        def_type = if wrapped isa Models.Wrapped
            typeof(wrapped.model.definition)
        else
            typeof(wrapped)
        end
        status = isempty(tagged) ? "MISSING" : "ok"
        println("  [$status] path=$mp  tagged=$tagged  reified=$(nameof(def_type))  bindings=$(length(bindings)) keys")
    end
end


=== repressilator.schedule.json ===
  [ok] path=+.do  tagged=[Symbol("{regulation/v1}")]  reified=Definition  bindings=1 keys

=== simple.schedule.json ===
  [ok] path=-2.do  tagged=[Symbol("{regulation/v1}")]  reified=Definition  bindings=0 keys

=== ACDC.schedule.json ===
  [ok] path=-2-1/1.do  tagged=[Symbol("{regulation/v1}")]  reified=Definition  bindings=0 keys
  [ok] path=-2-1/10.do  tagged=[Symbol("{regulation/v1}")]  reified=Definition  bindings=0 keys
  [ok] path=-2-1/11.do  tagged=[Symbol("{regulation/v1}")]  reified=Definition  bindings=0 keys
  [ok] path=-2-1/12.do  tagged=[Symbol("{regulation/v1}")]  reified=Definition  bindings=0 keys
  [ok] path=-2-1/13.do  tagged=[Symbol("{regulation/v1}")]  reified=Definition  bindings=0 keys
  [ok] path=-2-1/14.do  tagged=[Symbol("{regulation/v1}")]  reified=Definition  bindings=0 keys
  [ok] path=-2-1/15.do  tagged=[Symbol("{regulation/v1}")]  reified=Definition  bindings=0 keys
  [ok] path=-2-1/16.do  tagged=[Symbol("{regulation/v

In [26]:
# Surgical edit demo: add a gene to repressilator, preserving $ refs
raw = JSON.parsefile("examples/toy/repressilator.schedule.json", dicttype=Dict{Symbol, Any})
model_path = "+.do"

println("=== BEFORE ===")
println(JSON.json(raw; pretty=4))

# Navigate to the model block
model_block, _bindings = navigate_spec(raw, model_path)
v1_key = only(filter(k -> occursin(r"\{.*\}", String(k)), collect(keys(model_block))))
genes = model_block[v1_key][:genes]

# Add a 4th gene using $ ref for base_rates (preserves template references)
push!(genes, Dict{Symbol, Any}(
    Symbol("\$") => ["defaults", "gene"],
    :activation => [Dict{Symbol, Any}(:from => 1, :at => 5.0)],
))

modified_json = JSON.json(raw; pretty=4)
println("\n=== AFTER ===")
println(modified_json)

# Verify $ refs preserved
println("\n=== VERIFICATION ===")
for (i, g) in enumerate(genes)
    has_ref = haskey(g, Symbol("\$"))
    println("  Gene $i: \$ ref=$(has_ref)")
end

# Round-trip: reload, reify, check genes
sched2 = Models.parse(modified_json, seed="seed123")
def2 = Scheduling.reify(sched2, model_path).model.definition
println("\nReloaded: $(length(def2.genes)) genes")
for g in def2.genes
    println("  $(g.name): act=$(length(g.activation.slots)) rep=$(length(g.repression.slots)) prot=$(length(g.proteolysis.slots))")
end

=== BEFORE ===
{
    "step": {
        "do": {
            "{regulation/v1}": {
                "genes": [
                    {
                        "activation": [
                            {
                                "from": 1,
                                "at": 2.0
                            }
                        ],
                        "repression": [
                            {
                                "from": 3,
                                "at": 4.0
                            }
                        ],
                        "$": [
                            "defaults",
                            "gene"
                        ]
                    },
                    {
                        "activation": [
                            {
                                "from": 2,
                                "at": 2.0
                            }
                        ],
                        "repression": [
                  

In [27]:
# Surgical edit on cascade (uses local $ refs like "$": "gene")
raw_cascade = JSON.parsefile("examples/toy/cascade.schedule.json", dicttype=Dict{Symbol, Any})
cascade_mp = "+.do"  # known from previous test

println("=== BEFORE ===")
println(JSON.json(raw_cascade; pretty=4))

# Navigate and edit
model_block, _bindings = navigate_spec(raw_cascade, cascade_mp)
v1_key = only(filter(k -> occursin(r"\{.*\}", String(k)), collect(keys(model_block))))
genes = model_block[v1_key][:genes]

# Add gene using local $ ref to "gene" (defined in the outer Scope)
push!(genes, Dict{Symbol, Any}(
    Symbol("\$") => "gene",
    :activation => [Dict{Symbol, Any}(:from => 1, :at => 3.0)],
))

modified = JSON.json(raw_cascade; pretty=4)
println("\n=== AFTER ===")
println(modified)

# Round-trip
sched2 = Models.parse(modified, seed="seed123")
def2 = Scheduling.reify(sched2, cascade_mp).model.definition
println("\n=== VERIFICATION: $(length(def2.genes)) genes ===")
for g in def2.genes
    println("  $(g.name): act=$(length(g.activation.slots)) rep=$(length(g.repression.slots))")
end

=== BEFORE ===
{
    "gene": {
        "base_rates": {
            "activation": 0.027,
            "processing": 0.02,
            "transcription": 1.0,
            "protein_decay": 1.0e-6,
            "trigger": 6.6e-8,
            "mrna_decay": 0.0011,
            "deactivation": 0.0023,
            "translation": 5.0e-6,
            "abortion": 0.001,
            "premrna_decay": 0.001
        }
    },
    "step": {
        "do": {
            "{regulation/v1}": {
                "genes": [
                    {
                        "base_rates": {
                            "protein_decay": 1.0e-7,
                            "trigger": 6.6e-9,
                            "deactivation": 10.0,
                            "translation": 5.0e-7,
                            "$": [
                                "gene",
                                "base_rates"
                            ]
                        }
                    },
                    {
                

In [44]:
# ACDC: many execution_paths but one model_path (nested each loops = parameter sweep)
sched_acdc = Models.load("examples/toy/ACDC.schedule.json", seed="seed123")

execution_paths = String[]
model_paths = Set{String}()
for p in collect_primitives(sched_acdc)
    isfinite(p.dt) && p.dt > 0.0 || continue
    push!(execution_paths, p.path)
    push!(model_paths, p.primitive_path)
end

println("Execution paths: $(length(execution_paths)) (from 5x17 each-loop sweep)")
println("Unique model_paths: $(model_paths)")
println()
println("The nested each loops iterate over parameter values for edges,")
println("but the V1 model definition is the SAME dict in the JSON.")
println("So we only need to navigate to the one model_path to edit it.")

# Show before/after
raw_acdc = JSON.parsefile("examples/toy/ACDC.schedule.json", dicttype=Dict{Symbol, Any})


Execution paths: 17000 (from 5x17 each-loop sweep)
Unique model_paths: Set(["-2-4/1.do", "-2-5/7.do", "-2-4/3.do", "-2-1/12.do", "-2-4/13.do", "-2-4/5.do", "-2-4/11.do", "-2-1/7.do", "-2-2/11.do", "-2-3/14.do", "-2-2/5.do", "-2-5/5.do", "-2-1/3.do", "-2-4/8.do", "-2-4/2.do", "-2-4/9.do", "-2-5/4.do", "-2-3/1.do", "-2-3/11.do", "-2-1/9.do", "-2-1/5.do", "-2-3/4.do", "-2-2/8.do", "-2-1/4.do", "-2-3/17.do", "-2-1/1.do", "-2-3/12.do", "-2-4/4.do", "-2-2/16.do", "-2-2/15.do", "-2-3/3.do", "-2-3/15.do", "-2-3/16.do", "-2-1/8.do", "-2-2/12.do", "-2-5/1.do", "-2-5/9.do", "-2-1/16.do", "-2-3/9.do", "-2-1/6.do", "-2-2/3.do", "-2-4/10.do", "-2-1/11.do", "-2-5/12.do", "-2-5/2.do", "-2-4/15.do", "-2-5/10.do", "-2-2/4.do", "-2-3/6.do", "-2-3/8.do", "-2-2/13.do", "-2-2/9.do", "-2-5/15.do", "-2-3/13.do", "-2-2/17.do", "-2-5/8.do", "-2-1/13.do", "-2-2/2.do", "-2-3/10.do", "-2-2/1.do", "-2-2/10.do", "-2-5/16.do", "-2-2/14.do", "-2-1/10.do", "-2-1/14.do", "-2-4/14.do", "-2-1/2.do", "-2-5/11.do", "-2-1/17

2-element Vector{Any}:
 Dict{Symbol, Any}(Symbol("{add}") => Dict{Symbol, Any}(Symbol("3.proteins") => 20, Symbol("1.proteins") => 20, Symbol("2.proteins") => 20, :$ => Any["defaults", "bootstrap"]))
 Dict{Symbol, Any}(:as => "2->3", :step => Dict{Symbol, Any}(:branch => true, :as => "1->3", :step => Dict{Symbol, Any}(:do => Dict{Symbol, Any}(Symbol("{regulation/v1}") => Dict{Symbol, Any}(:genes => Any[Dict{Symbol, Any}(:activation => Any[Dict{Symbol, Any}(:from => "1", :at => 2.0)], :repression => Any[Dict{Symbol, Any}(:from => "3", :at => 4.0)], :$ => Any["defaults", "gene"]), Dict{Symbol, Any}(:activation => Any[Dict{Symbol, Any}(:from => "2", :at => 2.0)], :repression => Any[Dict{Symbol, Any}(:from => "1", :at => 4.0)], :$ => Any["defaults", "gene"]), Dict{Symbol, Any}(:activation => Any[Dict{Symbol, Any}(:from => "3", :at => 2.0)], :repression => Any[Dict{Symbol, Any}(:from => "1", :at => Dict{Symbol, Any}(:$ => "1->3")), Dict{Symbol, Any}(:from => "2", :at => Dict{Symbol, Any}(:$

In [38]:
# All 85 model_paths are different strings but navigate to the SAME dict.
# Verify this: navigate each path and check they return the same object (===)
raw_acdc = JSON.parsefile("examples/toy/ACDC.schedule.json", dicttype=Dict{Symbol, Any})
targets = [first(navigate_spec(raw_acdc, mp)) for mp in model_paths]
println("All $(length(targets)) paths navigate to same dict: $(all(t -> t === targets[1], targets))")

# Use any path (they're all equivalent for navigation)
acdc_mp = first(model_paths)

println("\n=== BEFORE ===")
println(JSON.json(raw_acdc; pretty=4))

model_block, _bindings = navigate_spec(raw_acdc, acdc_mp)
v1_key = only(filter(k -> occursin(r"\{.*\}", String(k)), collect(keys(model_block))))
genes = model_block[v1_key][:genes]

# Add 4th gene
push!(genes, Dict{Symbol, Any}(
    Symbol("\$") => ["defaults", "gene"],
    :repression => [Dict{Symbol, Any}(:from => 2, :at => 6.0)],
))

modified = JSON.json(raw_acdc; pretty=4)
println("\n=== AFTER ===")
println(modified)

# Round-trip
sched2 = Models.parse(modified, seed="seed123")
def2 = Scheduling.reify(sched2, acdc_mp).model.definition
println("\n=== VERIFICATION: $(length(def2.genes)) genes ===")
for g in def2.genes
    println("  $(g.name): act=$(length(g.activation.slots)) rep=$(length(g.repression.slots))")
end

All 85 paths navigate to same dict: true

=== BEFORE ===
[
    {
        "{add}": {
            "3.proteins": 20,
            "1.proteins": 20,
            "2.proteins": 20,
            "$": [
                "defaults",
                "bootstrap"
            ]
        }
    },
    {
        "as": "2->3",
        "step": {
            "branch": true,
            "as": "1->3",
            "step": {
                "do": {
                    "{regulation/v1}": {
                        "genes": [
                            {
                                "activation": [
                                    {
                                        "from": "1",
                                        "at": 2.0
                                    }
                                ],
                                "repression": [
                                    {
                                        "from": "3",
                                        "at": 4.0
                 

# Final approach: surgical schedule editing

**Pipeline** (server endpoint will do this):
1. Parse JSON to `Dict{Symbol, Any}` (preserves raw structure incl. `$` refs)
2. Collect model_paths via dryrun: `schedule(FlatState(); dryrun=...)`, gather `primitive!.path`
3. `navigate_spec(raw, model_path)` to reach the model block
4. Find the tagged key (`{regulation/v1}` etc.) to get the genes array
5. Surgically edit (add gene, rename gene, add/remove edge) — mutate in place
6. `JSON.json(raw; pretty=4)` to serialise back
7. `Models.parse(json)` to reload and verify

**Helper functions** needed in server code:
- `navigate_spec(spec, path)` — dict navigation by model_path
- `find_v1_genes(block)` — find `{regulation/v1}` key, return genes array
- `add_gene!(genes; ref)` — push new gene dict with `$` ref
- `rename_gene!(genes, old, new)` — update name + all `from` references
- `add_edge!(genes, type, from, to, params)` — push edge into gene's edge array
- `remove_edge!(genes, type, from, to)` — splice edge out

In [9]:
# --- Final helper functions ---

"""Find the tagged template key (e.g. Symbol("{regulation/v1}")) in a model block dict."""
function find_tagged_key(block::AbstractDict{Symbol})
    for k in keys(block)
        occursin(r"^\{.*\}$", String(k)) && return k
    end
    error("No tagged template key found. Keys: $(keys(block))")
end

"""
    find_v1_genes(block, bindings) -> Vector (the genes array to mutate)

Get the genes array from a model block, resolving `\$` references if needed.
Uses `pluck` from specifications.jl — the same mechanism as `substituted()`.
"""
function find_v1_genes(block::AbstractDict{Symbol}, bindings::AbstractDict{Symbol})
    v1_block = block[find_tagged_key(block)]
    source = resolve_dollar(v1_block, bindings)
    return source[:genes]
end

"""Collect unique model_paths from a schedule via dryrun."""
function collect_model_paths(schedule::Scheduling.Schedule)
    paths = Set{String}()
    schedule(Models.FlatState(); dryrun = (primitive!, x, dt; path, _...) -> begin
        isfinite(dt) && dt > 0.0 && push!(paths, primitive!.path)
    end)
    return paths
end

"""
    edit_schedule(raw_spec, model_path, edit_fn!)

Navigate to the model block at `model_path` (accumulating scope bindings),
resolve `\$` refs to find the actual genes array, call `edit_fn!(genes)`,
then return the modified JSON string.
"""
function edit_schedule(raw_spec, model_path::AbstractString, edit_fn!)
    block, bindings = navigate_spec(raw_spec, model_path)
    genes = find_v1_genes(block, bindings)
    edit_fn!(genes)
    return JSON.json(raw_spec; pretty=4)
end

edit_schedule

In [10]:
# Debug: trace what navigate_spec actually returns for repressilator
raw = JSON.parsefile("examples/toy/repressilator.schedule.json", dicttype=Dict{Symbol, Any})
result = navigate_spec(raw, "+.do")
println("navigate_spec return type: ", typeof(result))
println("result: ", result)
println()

# Now check what edit_schedule does step by step
block, bindings = result
println("block type: ", typeof(block))
println("bindings type: ", typeof(bindings))
println()

# Check find_v1_genes
println("find_v1_genes args: block=$(typeof(block)), bindings=$(typeof(bindings))")
genes = find_v1_genes(block, bindings)
println("genes: $(length(genes)) genes")
println()

# Check which methods exist for find_v1_genes
println("Methods of find_v1_genes:")
display(methods(find_v1_genes))
println()

# Check which methods exist for edit_schedule
println("Methods of edit_schedule:")
display(methods(edit_schedule))

navigate_spec return type: Tuple{Dict{Symbol, Any}, Dict{Symbol, Any}}
result: (Dict{Symbol, Any}(Symbol("{regulation/v1}") => Dict{Symbol, Any}(:genes => Any[Dict{Symbol, Any}(:activation => Any[Dict{Symbol, Any}(:from => 1, :at => 2.0)], :repression => Any[Dict{Symbol, Any}(:from => 3, :at => 4.0)], :$ => Any["defaults", "gene"]), Dict{Symbol, Any}(:activation => Any[Dict{Symbol, Any}(:from => 2, :at => 2.0)], :repression => Any[Dict{Symbol, Any}(:from => 1, :at => 4.0)], :$ => Any["defaults", "gene"]), Dict{Symbol, Any}(:activation => Any[Dict{Symbol, Any}(:from => 3, :at => 2.0)], :repression => Any[Dict{Symbol, Any}(:from => 2, :at => 4.0)], :$ => Any["defaults", "gene"])])), Dict{Symbol, Any}(:label => "repressilator-plain"))

block type: Dict{Symbol, Any}
bindings type: Dict{Symbol, Any}

find_v1_genes args: block=Dict{Symbol, Any}, bindings=Dict{Symbol, Any}
genes: 3 genes

Methods of find_v1_genes:

Methods of edit_schedule:


# 1 method for generic function "find_v1_genes" from Main:
 [1] find_v1_genes(block::AbstractDict{Symbol}, bindings::AbstractDict{Symbol})
     @ ~/Code/GeneRegulatorySystems.jl/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X61sZmlsZQ==.jl:17

# 1 method for generic function "edit_schedule" from Main:
 [1] edit_schedule(raw_spec, model_path::AbstractString, edit_fn!)
     @ ~/Code/GeneRegulatorySystems.jl/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X61sZmlsZQ==.jl:39

In [11]:
# Comprehensive test: navigate + resolve + edit + round-trip on ALL example schedules
all_schedules = [
    # toy
    "examples/toy/repressilator.schedule.json",
    "examples/toy/repressilator-entrained.schedule.json",
    "examples/toy/repressilator-intervened.schedule.json",
    "examples/toy/switch.schedule.json",
    "examples/toy/cascade.schedule.json",
    "examples/toy/bursts.schedule.json",
    "examples/toy/ACDC.schedule.json",
    "examples/toy/ACDC-simple.schedule.json",
    "examples/toy/coupled-repressilator.schedule.json",
    "examples/toy/steady-state-variance.schedule.json",
    "examples/toy/multifate2-dimer.schedule.json",
    "examples/toy/multifate2-proteolytic.schedule.json",
    "examples/toy/multifate3-proteolytic.schedule.json",
    "examples/toy/sequential-models.schedule.json",
    "examples/toy/v1-to-differentiation.schedule.json",
    "examples/toy/differentiation-tree.schedule.json",
    # specification
    "examples/specification/simple.schedule.json",
    "examples/specification/differentiation.schedule.json",
    "examples/specification/kronecker.schedule.json",
    "examples/specification/prokaryote.schedule.json",
    "examples/specification/reactions.schedule.json",
    "examples/specification/minimal.schedule.json",
    "examples/specification/copies.schedule.json",
    "examples/specification/channels.schedule.json",
    "examples/specification/templating.schedule.json",
    "examples/specification/extraction.schedule.json",
    "examples/specification/random-differentiation.schedule.json",
    "examples/specification/aggregations.schedule.json",
    # benchmark (skip huge)
    "examples/benchmark/kronecker-tiny.schedule.json",
    "examples/benchmark/kronecker-small.schedule.json",
    "examples/benchmark/kronecker-medium.schedule.json",
    "examples/benchmark/kronecker-big.schedule.json",
    "examples/benchmark/differentiated-small.schedule.json",
    "examples/benchmark/differentiated-medium.schedule.json",
]

println("Testing $(length(all_schedules)) schedules\n")

for path in all_schedules
    name = basename(path)
    print("$name: ")

    sched = Models.load(path, seed="seed123")
    model_paths = collect_model_paths(sched)

    if isempty(model_paths)
        println("SKIP (no model primitives)")
        continue
    end

    raw = JSON.parsefile(path, dicttype=Dict{Symbol, Any})

    # Pick one representative model_path
    mp = first(model_paths)

    # Navigate (now returns bindings too)
    block, bindings = navigate_spec(raw, mp)

    # Find tagged key
    tagged = nothing
    if block isa AbstractDict{Symbol}
        for k in keys(block)
            if occursin(r"^\{.*\}$", String(k))
                tagged = k
                break
            end
        end
    end

    if tagged === nothing
        println("SKIP (no tagged key at path=$mp)")
        continue
    end

    model_type = String(tagged)

    # Only test surgical edit on V1 models
    if model_type != "{regulation/v1}"
        println("ok  navigate  model=$model_type  paths=$(length(model_paths))  (non-v1, skip edit)")
        continue
    end

    # Resolve through $ to find the actual genes source
    v1_inner = block[tagged]
    resolved = resolve_dollar(v1_inner, bindings)
    has_dollar = v1_inner !== resolved
    genes_before = length(resolved[:genes])

    # Surgical edit: add a gene via edit_schedule (which resolves $ internally)
    raw_copy = JSON.parsefile(path, dicttype=Dict{Symbol, Any})
    modified_json = edit_schedule(raw_copy, mp, genes -> push!(genes, Dict{Symbol, Any}(
        Symbol("\$") => ["defaults", "gene"],
        :activation => [Dict{Symbol, Any}(:from => 1, :at => 1.0)],
    )))

    # Reload and verify
    sched2 = Models.parse(modified_json, seed="seed123")
    wrapped = Scheduling.reify(sched2, mp)
    def = wrapped.model.definition
    genes_after = length(def.genes)

    status = genes_after == genes_before + 1 ? "ok" : "FAIL"
    ref_info = has_dollar ? "  [\$ resolved]" : ""
    println("$status  navigate+edit  model=$model_type  paths=$(length(model_paths))  genes=$genes_before->$genes_after$ref_info")
end

Testing 34 schedules

repressilator.schedule.json: ok  navigate+edit  model={regulation/v1}  paths=1  genes=3->4
repressilator-entrained.schedule.json: SKIP (no tagged key at path=++-4/3+-2.do)
repressilator-intervened.schedule.json: ok  navigate+edit  model={regulation/v1}  paths=1  genes=3->4
switch.schedule.json: ok  navigate+edit  model={regulation/v1}  paths=1  genes=2->3
cascade.schedule.json: ok  navigate+edit  model={regulation/v1}  paths=1  genes=4->5
bursts.schedule.json: ok  navigate+edit  model={regulation/v1}  paths=1  genes=2->3
ACDC.schedule.json: ok  navigate+edit  model={regulation/v1}  paths=85  genes=3->4
ACDC-simple.schedule.json: ok  navigate+edit  model={regulation/v1}  paths=1  genes=3->4
coupled-repressilator.schedule.json: ok  navigate+edit  model={regulation/v1}  paths=1  genes=6->7
steady-state-variance.schedule.json: ok  navigate+edit  model={regulation/v1}  paths=1  genes=2->3
multifate2-dimer.schedule.json: ok  navigate+edit  model={regulation/v1}  paths=1

ErrorException: mixing eukaryotic and prokaryotic genes is forbidden